In [3]:
#exploratory analysis in python 
#load the libraries and the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# load data
df = pd.read_csv("C:/Users/ahafeez7/Downloads/780/lm7_q03/highUtilizerY2_final.csv")

# quick look
df.head()

,patient_id,countClaims,highUtilizer,ELIX1,ELIX2,ELIX3,ELIX4,ELIX5,ELIX6,ELIX7,...,ELIX29,drug_early,drug_mid,drug_late,high_drug_use,proc_G12,proc_G13,proc_G14,proc_G15,proc_G16
0,PAT136795,10,0,0,0,0,0,0,0,0,...,0,1,1,1,0,0,1,0,0,0
1,PAT135068,3,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,PAT171574,23,0,0,0,0,0,0,0,0,...,0,1,1,1,1,0,1,0,0,0
3,PAT83536,16,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
4,PAT129222,151,1,0,0,0,0,0,1,0,...,0,1,0,1,1,1,1,0,0,0


In [24]:
# prepare the data
# target
y = df['highUtilizer']
# features
X = df.drop(columns=['highUtilizer', 'patient_id', 'countClaims'])

In [25]:
# handle categorial
X = pd.get_dummies(X, drop_first=True)

In [26]:
#training and testset split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [27]:
#scale data to get ready for logistic
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [28]:
#apply logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)
y_prob_log = log_model.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("AUC:", roc_auc_score(y_test, y_prob_log))

Logistic Regression
Accuracy: 0.923125
AUC: 0.7799141096249671


In [29]:
#apply decision tree
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=5, random_state=42)

tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)
y_prob_tree = tree_model.predict_proba(X_test)[:, 1]

print("\nDecision Tree")
print("Accuracy:", accuracy_score(y_test, y_pred_tree))
print("AUC:", roc_auc_score(y_test, y_prob_tree))


Decision Tree
Accuracy: 0.92375
AUC: 0.7106390418742898


In [30]:
#apply random forest
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("\nRandom Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("AUC:", roc_auc_score(y_test, y_prob_rf))


Random Forest
Accuracy: 0.92375
AUC: 0.7712157968353878


In [31]:
#compare models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf)
    ],
    'AUC': [
        roc_auc_score(y_test, y_prob_log),
        roc_auc_score(y_test, y_prob_tree),
        roc_auc_score(y_test, y_prob_rf)
    ]
})

print(results)

                 Model  Accuracy       AUC
0  Logistic Regression  0.923125  0.779914
1        Decision Tree  0.923750  0.710639
2        Random Forest  0.923750  0.771216


In [32]:
# check for data leakage if any then corr will sum up to 1
df[['countClaims','highUtilizer']].corr()

,countClaims,highUtilizer
countClaims,1.000000,0.756297
highUtilizer,0.756297,1.000000


In [33]:
#anotehr check
df.groupby('highUtilizer')['countClaims'].describe()

,count,mean,std,min,25%,50%,75%,max
highUtilizer,,,,,,,,
0,7378.0,29.195310,23.206505,1.0,11.00,23.0,42.00,99.0
1,622.0,174.808682,90.877988,100.0,118.25,145.0,193.75,885.0
